In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
# ============================================================
# CELL 1 — SETUP
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install mlflow -q

import mlflow
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

BASE       = '/content/drive/MyDrive/final_project/'
MODEL_DIR  = BASE + 'models/'
MLFLOW_DIR = BASE + 'mlruns/'

os.makedirs(MLFLOW_DIR, exist_ok=True)

mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
mlflow.set_experiment('customer-support-intelligence')

print("MLflow ready!")
print(f"Tracking URI: {MLFLOW_DIR}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
# ============================================================
# CELL 2 — DEFINE ALL EXPERIMENT RESULTS
# ============================================================
experiments = [
    # Baseline — Ticket Type
    {'model':'Logistic Regression', 'phase':'Baseline',
     'target':'Ticket Type',
     'accuracy':0.1992, 'f1_macro':0.1900,
     'f1_weighted':0.1891, 'roc_auc':0.4891,
     'params':{'C':1.0,'max_iter':1000,
               'class_weight':'balanced'}},

    {'model':'Naive Bayes', 'phase':'Baseline',
     'target':'Ticket Type',
     'accuracy':0.2173, 'f1_macro':0.2171,
     'f1_weighted':0.2168, 'roc_auc':0.5177,
     'params':{'alpha':0.1}},

    {'model':'Decision Tree', 'phase':'Baseline',
     'target':'Ticket Type',
     'accuracy':0.1952, 'f1_macro':0.0966,
     'f1_weighted':0.1124, 'roc_auc':0.5061,
     'params':{'max_depth':20,
               'class_weight':'balanced'}},

    # Advanced ML — Ticket Type
    {'model':'Random Forest', 'phase':'Advanced ML',
     'target':'Ticket Type',
     'accuracy':0.1953, 'f1_macro':0.1951,
     'f1_weighted':0.1950, 'roc_auc':0.4834,
     'params':{'n_estimators':200,'max_depth':30,
               'class_weight':'balanced'}},

    {'model':'XGBoost', 'phase':'Advanced ML',
     'target':'Ticket Type',
     'accuracy':0.2110, 'f1_macro':0.2109,
     'f1_weighted':0.2108, 'roc_auc':0.4895,
     'params':{'n_estimators':300,'max_depth':6,
               'learning_rate':0.1}},

    {'model':'LightGBM', 'phase':'Advanced ML',
     'target':'Ticket Type',
     'accuracy':0.2165, 'f1_macro':0.2164,
     'f1_weighted':0.2163, 'roc_auc':0.4992,
     'params':{'n_estimators':300,'max_depth':6,
               'learning_rate':0.1}},

    # Advanced ML — Ticket Priority
    {'model':'Random Forest', 'phase':'Advanced ML',
     'target':'Ticket Priority',
     'accuracy':0.2724, 'f1_macro':0.2718,
     'f1_weighted':0.2720, 'roc_auc':0.5062,
     'params':{'n_estimators':200,'max_depth':30,
               'class_weight':'balanced'}},

    {'model':'XGBoost', 'phase':'Advanced ML',
     'target':'Ticket Priority',
     'accuracy':0.2622, 'f1_macro':0.2602,
     'f1_weighted':0.2612, 'roc_auc':0.5023,
     'params':{'n_estimators':300,'max_depth':6,
               'learning_rate':0.1}},

    {'model':'LightGBM', 'phase':'Advanced ML',
     'target':'Ticket Priority',
     'accuracy':0.2614, 'f1_macro':0.2600,
     'f1_weighted':0.2610, 'roc_auc':0.4930,
     'params':{'n_estimators':300,'max_depth':6,
               'learning_rate':0.1}},

    # Transformer
    {'model':'DistilBERT', 'phase':'Transformer',
     'target':'Ticket Type',
     'accuracy':0.1943, 'f1_macro':0.1878,
     'f1_weighted':0.1878, 'roc_auc':None,
     'params':{'epochs':5,'lr':2e-5,
               'batch_size':32,'max_length':128}},

    # Regression
    {'model':'XGBoost Regressor', 'phase':'Regression',
     'target':'Time to Resolution',
     'rmse':43.4436, 'mae':34.3530, 'r2':-0.1182,
     'params':{'n_estimators':300,'max_depth':6,
               'learning_rate':0.1}},

    {'model':'LightGBM Regressor', 'phase':'Regression',
     'target':'Time to Resolution',
     'rmse':42.7499, 'mae':33.9086, 'r2':-0.0828,
     'params':{'n_estimators':300,'max_depth':6,
               'learning_rate':0.1}},

    # Clustering
    {'model':'K-Means', 'phase':'Clustering',
     'target':'Customer Segmentation',
     'silhouette':0.1362, 'davies_bouldin':1.8435,
     'params':{'k':5,'n_init':10}},

    # SHAP
    {'model':'XGBoost SHAP', 'phase':'Explainability',
     'target':'Ticket Type',
     'accuracy':0.1949, 'f1_macro':0.1949,
     'f1_weighted':0.1949, 'roc_auc':None,
     'params':{'n_estimators':300,'max_depth':6,
               'features':13}},
]

print(f"Total experiments to log: {len(experiments)}")

Total experiments to log: 14


In [3]:
# ============================================================
# CELL 3 — LOG ALL RUNS TO MLFLOW
# ============================================================
print("Logging all experiments...")
print("=" * 55)

for exp in experiments:
    run_name = f"{exp['model']} — {exp['target']}"

    with mlflow.start_run(run_name=run_name):
        # Log parameters
        mlflow.log_params(exp['params'])
        mlflow.log_param('phase',  exp['phase'])
        mlflow.log_param('target', exp['target'])
        mlflow.log_param('model',  exp['model'])

        # Log metrics based on task
        if exp['phase'] == 'Regression':
            mlflow.log_metric('rmse', exp['rmse'])
            mlflow.log_metric('mae',  exp['mae'])
            mlflow.log_metric('r2',   exp['r2'])

        elif exp['phase'] == 'Clustering':
            mlflow.log_metric('silhouette_score',
                              exp['silhouette'])
            mlflow.log_metric('davies_bouldin',
                              exp['davies_bouldin'])

        else:
            mlflow.log_metric('accuracy',
                              exp['accuracy'])
            mlflow.log_metric('f1_macro',
                              exp['f1_macro'])
            mlflow.log_metric('f1_weighted',
                              exp['f1_weighted'])
            if exp.get('roc_auc'):
                mlflow.log_metric('roc_auc',
                                  exp['roc_auc'])

        print(f"  ✓ {run_name}")

print(f"\nAll {len(experiments)} experiments logged!")

Logging all experiments...
  ✓ Logistic Regression — Ticket Type
  ✓ Naive Bayes — Ticket Type
  ✓ Decision Tree — Ticket Type
  ✓ Random Forest — Ticket Type
  ✓ XGBoost — Ticket Type
  ✓ LightGBM — Ticket Type
  ✓ Random Forest — Ticket Priority
  ✓ XGBoost — Ticket Priority
  ✓ LightGBM — Ticket Priority
  ✓ DistilBERT — Ticket Type
  ✓ XGBoost Regressor — Time to Resolution
  ✓ LightGBM Regressor — Time to Resolution
  ✓ K-Means — Customer Segmentation
  ✓ XGBoost SHAP — Ticket Type

All 14 experiments logged!


In [4]:
# ============================================================
# CELL 4 — LOG MODEL ARTIFACTS
# ============================================================
print("Logging model artifacts...")

model_files = {
    'tfidf_vectorizer'  : 'tfidf_vectorizer.pkl',
    'xgb_type'          : 'xgb_type.pkl',
    'xgb_priority'      : 'xgb_priority.pkl',
    'xgb_regressor'     : 'xgb_regressor.pkl',
    'rf_type'           : 'rf_type.pkl',
    'rf_priority'       : 'rf_priority.pkl',
    'lgbm_type'         : 'lgbm_type.pkl',
    'lgbm_priority'     : 'lgbm_priority.pkl',
    'kmeans_model'      : 'kmeans_model.pkl',
    'le_type'           : 'le_type.pkl',
    'le_priority'       : 'le_priority.pkl',
    'standard_scaler'   : 'standard_scaler.pkl',
    'xgb_shap'          : 'xgb_shap.pkl',
}

with mlflow.start_run(run_name="Production Models — Artifacts"):
    mlflow.log_param('description', 'All saved model artifacts')
    mlflow.log_param('api_url',
        'https://unnatural-whole-geometry.ngrok-free.dev')
    mlflow.log_param('total_models', len(model_files))

    for name, filename in model_files.items():
        path = MODEL_DIR + filename
        if os.path.exists(path):
            mlflow.log_artifact(path, artifact_path='models')
            print(f"  ✓ {filename}")
        else:
            print(f"  ✗ {filename} — not found")

    # Log plots
    plots_dir = BASE + 'plots/'
    if os.path.exists(plots_dir):
        for plot_file in sorted(os.listdir(plots_dir)):
            if plot_file.endswith('.png'):
                mlflow.log_artifact(
                    plots_dir + plot_file,
                    artifact_path='plots')
        print(f"  ✓ All plots logged")

print("\nArtifacts logged!")

Logging model artifacts...
  ✓ tfidf_vectorizer.pkl
  ✓ xgb_type.pkl
  ✓ xgb_priority.pkl
  ✓ xgb_regressor.pkl
  ✓ rf_type.pkl
  ✓ rf_priority.pkl
  ✓ lgbm_type.pkl
  ✓ lgbm_priority.pkl
  ✓ kmeans_model.pkl
  ✓ le_type.pkl
  ✓ le_priority.pkl
  ✓ standard_scaler.pkl
  ✓ xgb_shap.pkl
  ✓ All plots logged

Artifacts logged!


In [5]:
# ============================================================
# CELL 5 — VIEW EXPERIMENT SUMMARY
# ============================================================
runs = mlflow.search_runs(
    experiment_names=['customer-support-intelligence'],
    order_by=['metrics.f1_macro DESC']
)

print("=" * 65)
print("MLFLOW EXPERIMENT SUMMARY")
print("=" * 65)
print(f"Total runs logged: {len(runs)}")

# Classification runs
clf_runs = runs[runs['metrics.f1_macro'].notna()].copy()
clf_runs = clf_runs[['tags.mlflow.runName',
                      'metrics.accuracy',
                      'metrics.f1_macro',
                      'metrics.roc_auc']].copy()
clf_runs.columns = ['Run','Accuracy','F1 Macro','ROC-AUC']
clf_runs = clf_runs.sort_values('F1 Macro', ascending=False)

print("\nCLASSIFICATION RUNS:")
print(clf_runs.to_string(index=False))

# Regression runs
reg_runs = runs[runs['metrics.rmse'].notna()].copy()
reg_runs = reg_runs[['tags.mlflow.runName',
                      'metrics.rmse',
                      'metrics.mae',
                      'metrics.r2']].copy()
reg_runs.columns = ['Run','RMSE','MAE','R2']

print("\nREGRESSION RUNS:")
print(reg_runs.to_string(index=False))

# Clustering runs
cl_runs = runs[runs['metrics.silhouette_score'].notna()].copy()
cl_runs = cl_runs[['tags.mlflow.runName',
                    'metrics.silhouette_score',
                    'metrics.davies_bouldin']].copy()
cl_runs.columns = ['Run','Silhouette','Davies-Bouldin']

print("\nCLUSTERING RUNS:")
print(cl_runs.to_string(index=False))

print(f"\nRequirement: minimum 10 runs ✓ ({len(runs)} logged)")

MLFLOW EXPERIMENT SUMMARY
Total runs logged: 105

CLASSIFICATION RUNS:
                              Run  Accuracy  F1 Macro  ROC-AUC
  Random Forest — Ticket Priority    0.2724    0.2718   0.5062
  Random Forest — Ticket Priority    0.2724    0.2718   0.5062
  Random Forest — Ticket Priority    0.2724    0.2718   0.5062
  Random Forest — Ticket Priority    0.2724    0.2718   0.5062
  Random Forest — Ticket Priority    0.2724    0.2718   0.5062
  Random Forest — Ticket Priority    0.2724    0.2718   0.5062
  Random Forest — Ticket Priority    0.2724    0.2718   0.5062
        XGBoost — Ticket Priority    0.2622    0.2602   0.5023
        XGBoost — Ticket Priority    0.2622    0.2602   0.5023
        XGBoost — Ticket Priority    0.2622    0.2602   0.5023
        XGBoost — Ticket Priority    0.2622    0.2602   0.5023
        XGBoost — Ticket Priority    0.2622    0.2602   0.5023
        XGBoost — Ticket Priority    0.2622    0.2602   0.5023
        XGBoost — Ticket Priority    0.2622    

In [6]:
# ============================================================
# CELL 6 — FINAL PROJECT SUMMARY
# ============================================================
print("""
╔══════════════════════════════════════════════════════════════╗
║      AI CUSTOMER SUPPORT INTELLIGENCE PLATFORM             ║
║                  PROJECT COMPLETE ✓                        ║
╚══════════════════════════════════════════════════════════════╝

NOTEBOOKS:
  01_EDA.ipynb              ✓ 9 plots saved
  02_Preprocessing.ipynb    ✓ TF-IDF + BERT + features
  03_Baseline_Models.ipynb  ✓ LR, Naive Bayes, Decision Tree
  04_Advanced_Models.ipynb  ✓ RF, XGBoost, LightGBM
  05_DistilBERT.ipynb       ✓ GPU fine-tuning T4
  06_Clustering.ipynb       ✓ K-Means K=5
  07_SHAP.ipynb             ✓ 13 features explained
  08_FastAPI.ipynb          ✓ Live API endpoint
  09_MLflow.ipynb           ✓ 15 experiments logged

BEST MODELS:
  Ticket Type     : LightGBM      F1: 0.2164
  Ticket Priority : Random Forest F1: 0.2718
  Regression      : LightGBM      RMSE: 42.75h
  DistilBERT      : Val F1: 0.19  (synthetic data limit)
  Clustering      : K=5           Silhouette: 0.1362

LIVE API:
  URL   : https://unnatural-whole-geometry.ngrok-free.dev
  Docs  : https://unnatural-whole-geometry.ngrok-free.dev/docs
""")


╔══════════════════════════════════════════════════════════════╗
║      AI CUSTOMER SUPPORT INTELLIGENCE PLATFORM             ║
║                  PROJECT COMPLETE ✓                        ║
╚══════════════════════════════════════════════════════════════╝

NOTEBOOKS:
  01_EDA.ipynb              ✓ 9 plots saved
  02_Preprocessing.ipynb    ✓ TF-IDF + BERT + features
  03_Baseline_Models.ipynb  ✓ LR, Naive Bayes, Decision Tree
  04_Advanced_Models.ipynb  ✓ RF, XGBoost, LightGBM
  05_DistilBERT.ipynb       ✓ GPU fine-tuning T4
  06_Clustering.ipynb       ✓ K-Means K=5
  07_SHAP.ipynb             ✓ 13 features explained
  08_FastAPI.ipynb          ✓ Live API endpoint
  09_MLflow.ipynb           ✓ 15 experiments logged

BEST MODELS:
  Ticket Type     : LightGBM      F1: 0.2164
  Ticket Priority : Random Forest F1: 0.2718
  Regression      : LightGBM      RMSE: 42.75h
  DistilBERT      : Val F1: 0.19  (synthetic data limit)
  Clustering      : K=5           Silhouette: 0.1362

LIVE API:
  UR